# Feature Engineering

The goal of this notebook is to evaluate whether additional
leakage-safe features improve the Logistic Regression baseline.

Each candidate feature is first evaluated independently using the
same temporal training and validation periods defined in the baseline
experiment.

Average Precision is used as the primary comparison metric because
the target is highly imbalanced.

In [1]:
import pandas as pd
import numpy as np
from typing import List

from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.metrics import (precision_score, recall_score,
                             average_precision_score,
                             accuracy_score, f1_score)

In [2]:
RANDOM_SEED = 42
df = pd.read_csv('../data/Synthetic_Financial_datasets_log.csv')

In [3]:
def data_split(
        new_features=None,
        data=None,
        base_features=None
):
    if data is None:
        data = df

    if base_features is None:
        base_features = ['step', 'type', 'amount', 'isFraud']

    features = base_features.copy()

    if new_features is not None:
        features.extend(new_features)

    temp_df = data[features].copy()

    test_point = 550
    calibration_point = test_point - 50
    validation_point = calibration_point - 100

    validation = temp_df[
        (temp_df['step'] <= calibration_point)
        & (temp_df['step'] > validation_point)
    ].copy()

    train = temp_df[
        temp_df['step'] <= validation_point
    ].copy()

    X_train = train.drop('isFraud', axis=1)
    y_train = train['isFraud']

    X_val = validation.drop('isFraud', axis=1)
    y_val = validation['isFraud']

    return X_train, y_train, X_val, y_val

In [4]:
def get_preprocessor(
        numeric_features:List[str],
        categorical_features:List[str],
        other_features:List[str]
):
    preprocessor = ColumnTransformer(transformers=[
        ('numeric', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(
            drop='first',
            sparse_output=False,
            handle_unknown='ignore'
        ), categorical_features),
        ('keep', 'passthrough', other_features)
    ], remainder='drop')

    return preprocessor


In [5]:
def get_proba_predictions(
        X_train,
        y_train,
        X_val,
        numeric_features,
        categorical_features,
        other_features=None
):

    if other_features is None:
        other_features = []

    preprocessor = get_preprocessor(
        numeric_features,
        categorical_features,
        other_features
    )
    logreg_unbalanced = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(class_weight=None,
                                     random_state=RANDOM_SEED))
    ])

    logreg_balanced = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', LogisticRegression(class_weight='balanced',
                                     random_state=RANDOM_SEED))
    ])

    logreg_unbalanced.fit(X_train, y_train)
    logreg_balanced.fit(X_train, y_train)

    y_proba_unbalanced = logreg_unbalanced.predict_proba(X_val)[:, 1]
    y_proba_balanced = logreg_balanced.predict_proba(X_val)[:, 1]

    return y_proba_unbalanced, y_proba_balanced


In [6]:
def model_estimation(y_true, y_proba):
    y_pred = (y_proba >= 0.5).astype(int)
    precision = precision_score(y_true, y_pred, zero_division=0)
    recall = recall_score(y_true, y_pred, zero_division=0)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    accuracy = accuracy_score(y_true, y_pred)
    average_precision = average_precision_score(y_true, y_proba)

    return {
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'accuracy': accuracy,
        'average_precision': average_precision,
    }

## Experimental strategy

To isolate the effect of each feature, candidate features are initially
added one at a time to the baseline feature set:

- `log_amount`
- `hour`
- `day`
- `dest_type`

Features that improve validation Average Precision will then be combined
in a final feature set.

In [7]:
unbalanced_res_table = pd.DataFrame(columns=[
    'Model',
    'precision',
    'recall',
    'f1',
    'accuracy',
    'average_precision',
])

balanced_res_table = pd.DataFrame(columns=[
    'Model',
    'precision',
    'recall',
    'f1',
    'accuracy',
    'average_precision',
])

In [8]:
X_train, y_train, X_val, y_val = data_split()
y_proba_unbalanced, y_proba_balanced = get_proba_predictions(
    X_train,
    y_train,
    X_val,
    numeric_features=['step', 'amount'],
    categorical_features=['type']
)

metrics_unbalanced = model_estimation(y_val, y_proba_unbalanced)
metrics_balanced = model_estimation(y_val, y_proba_balanced)

metrics_unbalanced['Model'] = 'Unweighted Logistic Regression (base)'
metrics_balanced['Model'] = 'Balanced Logistic Regression (base)'

unbalanced_res_table = pd.concat([unbalanced_res_table,
                                  pd.DataFrame([metrics_unbalanced])],
                                 ignore_index=True)
balanced_res_table = pd.concat([balanced_res_table,
                                pd.DataFrame([metrics_balanced])],
                               ignore_index=True)

## Logarithmic amount

In [9]:
df['log_amount'] = np.log1p(df['amount'])

X_train, y_train, X_val, y_val = data_split(new_features=['log_amount'])
y_proba_unbalanced, y_proba_balanced = get_proba_predictions(
    X_train,
    y_train,
    X_val,
    numeric_features=['step', 'amount', 'log_amount'],
    categorical_features=['type']
)

metrics_unbalanced = model_estimation(y_val, y_proba_unbalanced)
metrics_balanced = model_estimation(y_val, y_proba_balanced)

metrics_unbalanced['Model'] = 'Unweighted Logistic Regression (log_amount)'
metrics_balanced['Model'] = 'Balanced Logistic Regression (log_amount)'

unbalanced_res_table = pd.concat([unbalanced_res_table,
                                  pd.DataFrame([metrics_unbalanced])],
                                 ignore_index=True)
balanced_res_table = pd.concat([balanced_res_table,
                                pd.DataFrame([metrics_balanced])],
                               ignore_index=True)

## Hour

In [10]:
df['hour'] = (df['step'] - 1) % 24

X_train, y_train, X_val, y_val = data_split(new_features=['hour'])
y_proba_unbalanced, y_proba_balanced = get_proba_predictions(
    X_train,
    y_train,
    X_val,
    numeric_features=['step', 'amount'],
    categorical_features=['type'],
    other_features=['hour']
)

metrics_unbalanced = model_estimation(y_val, y_proba_unbalanced)
metrics_balanced = model_estimation(y_val, y_proba_balanced)

metrics_unbalanced['Model'] = 'Unweighted Logistic Regression (hour)'
metrics_balanced['Model'] = 'Balanced Logistic Regression (hour)'

unbalanced_res_table = pd.concat([unbalanced_res_table,
                                  pd.DataFrame([metrics_unbalanced])],
                                 ignore_index=True)
balanced_res_table = pd.concat([balanced_res_table,
                                pd.DataFrame([metrics_balanced])],
                               ignore_index=True)

## Day

In [11]:
df['day'] = (df['step'] - 1) // 24

X_train, y_train, X_val, y_val = data_split(new_features=['day'])
y_proba_unbalanced, y_proba_balanced = get_proba_predictions(
    X_train,
    y_train,
    X_val,
    numeric_features=['step', 'amount'],
    categorical_features=['type'],
    other_features=['day']
)

metrics_unbalanced = model_estimation(y_val, y_proba_unbalanced)
metrics_balanced = model_estimation(y_val, y_proba_balanced)

metrics_unbalanced['Model'] = 'Unweighted Logistic Regression (day)'
metrics_balanced['Model'] = 'Balanced Logistic Regression (day)'

unbalanced_res_table = pd.concat([unbalanced_res_table,
                                  pd.DataFrame([metrics_unbalanced])],
                                 ignore_index=True)
balanced_res_table = pd.concat([balanced_res_table,
                                pd.DataFrame([metrics_balanced])],
                               ignore_index=True)

## Dest type

In [12]:
df['dest_type'] = df['nameDest'].str[0]

X_train, y_train, X_val, y_val = data_split(new_features=['dest_type'])
y_proba_unbalanced, y_proba_balanced = get_proba_predictions(
    X_train,
    y_train,
    X_val,
    numeric_features=['step', 'amount'],
    categorical_features=['type', 'dest_type']
)

metrics_unbalanced = model_estimation(y_val, y_proba_unbalanced)
metrics_balanced = model_estimation(y_val, y_proba_balanced)

metrics_unbalanced['Model'] = 'Unweighted Logistic Regression (dest_type)'
metrics_balanced['Model'] = 'Balanced Logistic Regression (dest_type)'

unbalanced_res_table = pd.concat([unbalanced_res_table,
                                  pd.DataFrame([metrics_unbalanced])],
                                 ignore_index=True)
balanced_res_table = pd.concat([balanced_res_table,
                                pd.DataFrame([metrics_balanced])],
                               ignore_index=True)

## Sin Cos Hour

In [13]:
df['hour_sin'] = np.sin(2 * np.pi * df['hour'] / 24)
df['hour_cos'] = np.cos(2 * np.pi * df['hour'] / 24)

X_train, y_train, X_val, y_val = data_split(new_features=['hour_sin', 'hour_cos'])
y_proba_unbalanced, y_proba_balanced = get_proba_predictions(
    X_train,
    y_train,
    X_val,
    numeric_features=['step', 'amount'],
    categorical_features=['type'],
    other_features=['hour_sin', 'hour_cos']

)

metrics_unbalanced = model_estimation(y_val, y_proba_unbalanced)
metrics_balanced = model_estimation(y_val, y_proba_balanced)

metrics_unbalanced['Model'] = 'Unweighted Logistic Regression (hour_sin, hour_cos)'
metrics_balanced['Model'] = 'Balanced Logistic Regression (hour_sin, hour_cos)'

unbalanced_res_table = pd.concat([unbalanced_res_table,
                                  pd.DataFrame([metrics_unbalanced])],
                                 ignore_index=True)
balanced_res_table = pd.concat([balanced_res_table,
                                pd.DataFrame([metrics_balanced])],
                               ignore_index=True)

In [14]:
unbalanced_res_table

,Model,precision,recall,f1,accuracy,average_precision
0,Unweighted Logistic Regression (base),0.0,0.0,0.0,0.996055,0.06037
1,Unweighted Logistic Regression (log_amount),0.0,0.0,0.0,0.996055,0.093989
2,Unweighted Logistic Regression (hour),0.0,0.0,0.0,0.996055,0.194985
3,Unweighted Logistic Regression (day),0.0,0.0,0.0,0.996055,0.028323
4,Unweighted Logistic Regression (dest_type),0.0,0.0,0.0,0.996055,0.05404
5,"Unweighted Logistic Regression (hour_sin, hour...",0.0,0.0,0.0,0.996055,0.16047


In [15]:
balanced_res_table

,Model,precision,recall,f1,accuracy,average_precision
0,Balanced Logistic Regression (base),0.032614,0.733395,0.062451,0.91313,0.107476
1,Balanced Logistic Regression (log_amount),0.032804,0.72417,0.062765,0.91468,0.109994
2,Balanced Logistic Regression (hour),0.034706,0.817343,0.066584,0.909596,0.147237
3,Balanced Logistic Regression (day),0.034718,0.817343,0.066607,0.909629,0.147214
4,Balanced Logistic Regression (dest_type),0.032602,0.733395,0.062429,0.913097,0.107467
5,"Balanced Logistic Regression (hour_sin, hour_cos)",0.024641,0.828413,0.047859,0.869964,0.159716


In [16]:
X_train, y_train, X_val, y_val = data_split(new_features=[
    'log_amount',
    'day',
    'hour_sin',
    'hour_cos'
])
y_proba_unbalanced, y_proba_balanced = get_proba_predictions(
    X_train,
    y_train,
    X_val,
    numeric_features=['step', 'amount', 'log_amount'],
    categorical_features=['type'],
    other_features=['hour_sin', 'hour_cos', 'day']

)

metrics_unbalanced = model_estimation(y_val, y_proba_unbalanced)
metrics_balanced = model_estimation(y_val, y_proba_balanced)

metrics_unbalanced['Model'] = ('Unweighted Logistic Regression (log_amount, '
                               'day, hour_sin, hour_cos)')
metrics_balanced['Model'] = ('Balanced Logistic Regression (log_amount,'
                             'day, hour_sin, hour_cos)')

unbalanced_res_table = pd.concat([unbalanced_res_table,
                                  pd.DataFrame([metrics_unbalanced])],
                                 ignore_index=True)
balanced_res_table = pd.concat([balanced_res_table,
                                pd.DataFrame([metrics_balanced])],
                               ignore_index=True)

In [17]:
X_train, y_train, X_val, y_val = data_split(new_features=[
    'log_amount',
    'day',
    'hour_sin',
    'hour_cos'
])
y_proba_unbalanced, y_proba_balanced = get_proba_predictions(
    X_train,
    y_train,
    X_val,
    numeric_features=['step', 'amount', 'log_amount'],
    categorical_features=['type'],
    other_features=['hour_sin', 'hour_cos', 'day']
)

metrics_unbalanced = model_estimation(y_val, y_proba_unbalanced)
metrics_balanced = model_estimation(y_val, y_proba_balanced)

metrics_unbalanced['Model'] = 'Unweighted Logistic Regression (without hour)'
metrics_balanced['Model'] = 'Balanced Logistic Regression (without hour)'

unbalanced_res_table = pd.concat([unbalanced_res_table,
                                  pd.DataFrame([metrics_unbalanced])],
                                 ignore_index=True)
balanced_res_table = pd.concat([balanced_res_table,
                                pd.DataFrame([metrics_balanced])],
                               ignore_index=True)

In [18]:
X_train, y_train, X_val, y_val = data_split(new_features=[
    'log_amount',
    'hour_sin',
    'hour_cos'
])

y_proba_unbalanced, y_proba_balanced = get_proba_predictions(
    X_train,
    y_train,
    X_val,
    numeric_features=['step','amount', 'log_amount'],
    categorical_features=['type'],
    other_features=['hour_sin', 'hour_cos']

)

metrics_unbalanced = model_estimation(y_val, y_proba_unbalanced)
metrics_balanced = model_estimation(y_val, y_proba_balanced)

metrics_unbalanced['Model'] = 'Unweighted Logistic Regression (without day)'
metrics_balanced['Model'] = 'Balanced Logistic Regression (without day)'

unbalanced_res_table = pd.concat([unbalanced_res_table,
                                  pd.DataFrame([metrics_unbalanced])],
                                 ignore_index=True)
balanced_res_table = pd.concat([balanced_res_table,
                                pd.DataFrame([metrics_balanced])],
                               ignore_index=True)

In [19]:
X_train, y_train, X_val, y_val = data_split(new_features=[
    'log_amount',
    'hour_sin',
    'hour_cos',
    'day'
])

X_train.drop(columns='amount')
X_val.drop(columns='amount')

y_proba_unbalanced, y_proba_balanced = get_proba_predictions(
    X_train,
    y_train,
    X_val,
    numeric_features=['step', 'log_amount'],
    categorical_features=['type'],
    other_features=['hour_sin', 'hour_cos', 'day']

)

metrics_unbalanced = model_estimation(y_val, y_proba_unbalanced)
metrics_balanced = model_estimation(y_val, y_proba_balanced)

metrics_unbalanced['Model'] = 'Unweighted Logistic Regression (without amount)'
metrics_balanced['Model'] = 'Balanced Logistic Regression (without amount)'

unbalanced_res_table = pd.concat([unbalanced_res_table,
                                  pd.DataFrame([metrics_unbalanced])],
                                 ignore_index=True)
balanced_res_table = pd.concat([balanced_res_table,
                                pd.DataFrame([metrics_balanced])],
                               ignore_index=True)

In [20]:
X_train, y_train, X_val, y_val = data_split(new_features=[
    'log_amount',
    'day'
])

y_proba_unbalanced, y_proba_balanced = get_proba_predictions(
    X_train,
    y_train,
    X_val,
    numeric_features=['step', 'amount', 'log_amount'],
    categorical_features=['type'],
    other_features=['day']

)

metrics_unbalanced = model_estimation(y_val, y_proba_unbalanced)
metrics_balanced = model_estimation(y_val, y_proba_balanced)

metrics_unbalanced['Model'] = 'Unweighted Logistic Regression (without cyclical hour)'
metrics_balanced['Model'] = 'Balanced Logistic Regression (without cyclical hour)'

unbalanced_res_table = pd.concat([unbalanced_res_table,
                                  pd.DataFrame([metrics_unbalanced])],
                                 ignore_index=True)
balanced_res_table = pd.concat([balanced_res_table,
                                pd.DataFrame([metrics_balanced])],
                               ignore_index=True)

In [21]:
unbalanced_res_table

,Model,precision,recall,f1,accuracy,average_precision
0,Unweighted Logistic Regression (base),0.0,0.0,0.0,0.996055,0.06037
1,Unweighted Logistic Regression (log_amount),0.0,0.0,0.0,0.996055,0.093989
2,Unweighted Logistic Regression (hour),0.0,0.0,0.0,0.996055,0.194985
3,Unweighted Logistic Regression (day),0.0,0.0,0.0,0.996055,0.028323
4,Unweighted Logistic Regression (dest_type),0.0,0.0,0.0,0.996055,0.05404
5,"Unweighted Logistic Regression (hour_sin, hour...",0.0,0.0,0.0,0.996055,0.16047
6,"Unweighted Logistic Regression (log_amount, da...",0.0,0.0,0.0,0.996055,0.173795
7,Unweighted Logistic Regression (without hour),0.0,0.0,0.0,0.996055,0.173795
8,Unweighted Logistic Regression (without day),0.0,0.0,0.0,0.996055,0.218298
9,Unweighted Logistic Regression (without amount),0.0,0.0,0.0,0.996055,0.187034


In [22]:
balanced_res_table

,Model,precision,recall,f1,accuracy,average_precision
0,Balanced Logistic Regression (base),0.032614,0.733395,0.062451,0.91313,0.107476
1,Balanced Logistic Regression (log_amount),0.032804,0.72417,0.062765,0.91468,0.109994
2,Balanced Logistic Regression (hour),0.034706,0.817343,0.066584,0.909596,0.147237
3,Balanced Logistic Regression (day),0.034718,0.817343,0.066607,0.909629,0.147214
4,Balanced Logistic Regression (dest_type),0.032602,0.733395,0.062429,0.913097,0.107467
5,"Balanced Logistic Regression (hour_sin, hour_cos)",0.024641,0.828413,0.047859,0.869964,0.159716
6,"Balanced Logistic Regression (log_amount,day, ...",0.02846,0.831181,0.055036,0.8874,0.215117
7,Balanced Logistic Regression (without hour),0.02846,0.831181,0.055036,0.8874,0.215117
8,Balanced Logistic Regression (without day),0.024448,0.833948,0.047503,0.868064,0.160696
9,Balanced Logistic Regression (without amount),0.023915,0.809963,0.046459,0.868835,0.192809


## Feature Engineering Conclusions

The experiments show that temporal transformations provide the largest
improvement over the baseline feature set.

The balanced Logistic Regression baseline achieved an Average Precision
of approximately 0.107, while the final engineered feature set increased
Average Precision to approximately 0.215.

Ablation experiments confirmed that both the cyclical hour representation
and the `day` feature provide substantial predictive value. Keeping the
raw `amount` together with `log_amount` also improves performance compared
with using the logarithmic representation alone.

The final leakage-safe feature set consists of:

- `step`
- `type`
- `amount`
- `log_amount`
- `day`
- `hour_sin`
- `hour_cos`

This feature set will be used in the subsequent model comparison stage.